In [56]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import gudhi as gd
from gudhi import bottleneck_distance
from gudhi.wasserstein import wasserstein_distance
from persim import plot_diagrams
from matplotlib.backends.backend_pdf import PdfPages

#pull features from all players to compute persistent homology
def process_file(filepath):
    df = pd.read_csv(filepath)
    df.columns = df.iloc[0]
    df = df[1:].reset_index(drop=True)
    df = df.dropna(axis=1, how='all')
    df = df[pd.to_numeric(df["Week"], errors="coerce").notna()]
    df.columns.values[10] = 'Passing_Cmp'
    df.columns.values[11] = 'Passing_Att'
    df.columns.values[13] = 'First_Downs_Passing_%'
    df.columns.values[17] = 'Completed_Air_Yards_per_completion'
    df.columns.values[27] = 'Bad_Throw_%'
    df.columns.values[34] = 'Yards_per_Scramble'
    df.columns.values[35] = 'Rushing_Att'
    df.columns.values[36] = 'Rushing_Yds'
    drop_cols = ['Gcar', 'Gtm', 'Week', 'Date', 'Team', 'Result', 'GS']
    drop_cols = [c for c in drop_cols if c in df.columns]
    df = df.drop(columns=drop_cols)
    df = df.apply(pd.to_numeric, errors='coerce')
    df['Completion_Pct'] = df['Passing_Cmp'] / df['Passing_Att']
    df['Completion_Pct'] = df['Completion_Pct'].replace([np.inf, -np.inf], np.nan).fillna(0)
    features = [
        'Completion_Pct',
        'First_Downs_Passing_%',
        'Completed_Air_Yards_per_completion',
        'Bad_Throw_%',
        'Yards_per_Scramble',
        'Rushing_Att',
        'Rushing_Yds',
    ]
    features = [f for f in features if f in df.columns]

    # fill missing scramble and rushing stats with 0
    df['Yards_per_Scramble'] = df['Yards_per_Scramble'].fillna(0)
    df['Rushing_Att'] = df['Rushing_Att'].fillna(0)
    df['Rushing_Yds'] = df['Rushing_Yds'].fillna(0)

    point_cloud = df[features].dropna().to_numpy()

    rips = gd.RipsComplex(points=point_cloud)
    simplex_tree = rips.create_simplex_tree(max_dimension=2)
    simplex_tree.compute_persistence()
    diag = simplex_tree.persistence()
    H0 = np.array([p[1] for p in diag if p[0] == 0])
    H1 = np.array([p[1] for p in diag if p[0] == 1])

    # ensure correct shape
    H0 = H0.reshape(-1, 2) if H0.size > 0 else np.empty((0, 2))
    H1 = H1.reshape(-1, 2) if H1.size > 0 else np.empty((0, 2))

    return [H0, H1]


folder = os.path.join(os.path.expanduser("~"), "AMAT585 To Submit")
years = ["2018", "2019", "2020", "2021", "2022", "2023", "2024", "2025"]

all_distances = {
    ("Bottleneck", "H0"): [],
    ("Bottleneck", "H1"): [],
    ("Wasserstein", "H0"): [],
    ("Wasserstein", "H1"): [],
}

# go through process for each year
for year in years:
    output_folder = f"{year}_output"
    os.makedirs(output_folder, exist_ok=True)

    # compute persistent homology for each player
    results = {}
    for filename in os.listdir(folder):
        if year in filename and filename.endswith(".csv"):
            filepath = os.path.join(folder, filename)
            try:
                diagrams = process_file(filepath)
                results[filename] = diagrams
            except Exception:
                pass

    # save persistence diagrams as PDF for whole season
    pdf_path = os.path.join(output_folder, f"all_persistence_diagrams_{year}.pdf")
    with PdfPages(pdf_path) as pdf:
        for filename, diagrams in results.items():
            if all(d.size == 0 for d in diagrams):
                continue
            plt.figure()
            try:
                plot_diagrams(diagrams, show=False)
                plt.title(filename.replace(".csv", ""))
                pdf.savefig()
            except ValueError:
                pass
            finally:
                plt.close()

    # clean names from file names
    files = list(results.keys())
    clean_names = [f.replace(f"_{year}.csv", "") for f in files]
    name_map = dict(zip(files, clean_names))

    # compute bottleneck distances between all players
    H0_bn_matrix = pd.DataFrame(index=clean_names, columns=clean_names, dtype=float)
    H1_bn_matrix = pd.DataFrame(index=clean_names, columns=clean_names, dtype=float)
    for i, f1 in enumerate(files):
        H0_1, H1_1 = results[f1]
        name1 = name_map[f1]
        for j in range(i, len(files)):
            f2 = files[j]
            H0_2, H1_2 = results[f2]
            name2 = name_map[f2]
            H0_bn = bottleneck_distance(H0_1, H0_2)
            H1_bn = bottleneck_distance(H1_1, H1_2)
            H0_bn_matrix.loc[name1, name2] = H0_bn_matrix.loc[name2, name1] = H0_bn
            H1_bn_matrix.loc[name1, name2] = H1_bn_matrix.loc[name2, name1] = H1_bn

    # compute wasserstein distances between all players
    H0_ws_matrix = pd.DataFrame(index=clean_names, columns=clean_names, dtype=float)
    H1_ws_matrix = pd.DataFrame(index=clean_names, columns=clean_names, dtype=float)
    for i, f1 in enumerate(files):
        H0_1, H1_1 = results[f1]
        name1 = name_map[f1]
        for j in range(i, len(files)):
            f2 = files[j]
            H0_2, H1_2 = results[f2]
            name2 = name_map[f2]
            W_H0 = wasserstein_distance(H0_1, H0_2, order=1.0, internal_p=2.0, keep_essential_parts=False)
            W_H1 = wasserstein_distance(H1_1, H1_2, order=1.0, internal_p=2.0, keep_essential_parts=False)
            H0_ws_matrix.loc[name1, name2] = H0_ws_matrix.loc[name2, name1] = W_H0
            H1_ws_matrix.loc[name1, name2] = H1_ws_matrix.loc[name2, name1] = W_H1

    # save distance tables labeled by year
    H0_bn_matrix.to_csv(os.path.join(output_folder, f"H0_bottleneck_{year}.csv"))
    H1_bn_matrix.to_csv(os.path.join(output_folder, f"H1_bottleneck_{year}.csv"))
    H0_ws_matrix.to_csv(os.path.join(output_folder, f"H0_wasserstein_{year}.csv"))
    H1_ws_matrix.to_csv(os.path.join(output_folder, f"H1_wasserstein_{year}.csv"))

    # create per-year histograms
    for matrix, metric, dim_label in [
        (H0_bn_matrix, "Bottleneck", "H0"),
        (H1_bn_matrix, "Bottleneck", "H1"),
        (H0_ws_matrix, "Wasserstein", "H0"),
        (H1_ws_matrix, "Wasserstein", "H1")
    ]:
        values = matrix.to_numpy(dtype=float)
        values = values[np.triu_indices_from(values, k=1)]
        values = values[~np.isnan(values)]

        all_distances[(metric, dim_label)].extend(values.tolist())

        plt.figure()
        plt.hist(values, bins=20)
        plt.title(f"Histogram of {metric} Distances ({dim_label}) {year}")
        plt.xlabel("Distance")
        plt.ylabel("Frequency")
        plt.savefig(os.path.join(output_folder, f"{dim_label}_{metric}_hist_{year}.png"), dpi=300, bbox_inches='tight')
        plt.close()

# plot the combined histograms
combined_output = "combined_output"
os.makedirs(combined_output, exist_ok=True)

for (metric, dim_label), values in all_distances.items():
    values = np.array(values)
    plt.figure()
    plt.hist(values, bins=30)
    plt.title(f"Histogram of {metric} Distances ({dim_label}) 2018-2025")
    plt.xlabel("Distance")
    plt.ylabel("Frequency")
    plt.savefig(os.path.join(combined_output, f"{dim_label}_{metric}_hist_all_years.png"), dpi=300, bbox_inches='tight')
    plt.close()

# print notable H0 Wasserstein distances across all years
print("\n  Notable H0 Wasserstein Distances Across All Years ")
for year in years:
    ws_path = os.path.join(f"{year}_output", f"H0_wasserstein_{year}.csv")
    matrix = pd.read_csv(ws_path, index_col=0)
    for i, name1 in enumerate(matrix.index):
        for j, name2 in enumerate(matrix.columns):
            if j <= i:
                continue
            val = matrix.loc[name1, name2]
            try:
                val = float(val)
            except:
                continue
            if (0 < val < 15) or (val > 175):
                print(f"{year}: {name1} vs {name2} = {val:.4f}")

# print notable H1 Wasserstein distances across all years
print("\n Notable H1 Wasserstein Distances Across All Years ")
for year in years:
    ws_path = os.path.join(f"{year}_output", f"H1_wasserstein_{year}.csv")
    matrix = pd.read_csv(ws_path, index_col=0)
    for i, name1 in enumerate(matrix.index):
        for j, name2 in enumerate(matrix.columns):
            if j <= i:
                continue
            val = matrix.loc[name1, name2]
            try:
                val = float(val)
            except:
                continue
            if val > 8:
                print(f"{year}: {name1} vs {name2} = {val:.4f}")

# print notable H0 bottleneck distances across all years
print("\n Notable H0 Bottleneck Distances Across All Years ")
for year in years:
    bn_path = os.path.join(f"{year}_output", f"H0_bottleneck_{year}.csv")
    matrix = pd.read_csv(bn_path, index_col=0)
    for i, name1 in enumerate(matrix.index):
        for j, name2 in enumerate(matrix.columns):
            if j <= i:
                continue
            val = matrix.loc[name1, name2]
            try:
                val = float(val)
            except:
                continue
            if (0 < val < 3) or (val > 32):
                print(f"{year}: {name1} vs {name2} = {val:.4f}")

# print notable H1 bottleneck distances across all years
print("\n Notable H1 Bottleneck Distances Across All Years ")
for year in years:
    bn_path = os.path.join(f"{year}_output", f"H1_bottleneck_{year}.csv")
    matrix = pd.read_csv(bn_path, index_col=0)
    for i, name1 in enumerate(matrix.index):
        for j, name2 in enumerate(matrix.columns):
            if j <= i:
                continue
            val = matrix.loc[name1, name2]
            try:
                val = float(val)
            except:
                continue
            if val > 3.5:
                print(f"{year}: {name1} vs {name2} = {val:.4f}")

player_names = ["Purdy", "Mahomes", "Jackson", "Stafford", "Allen", "Lawrence", "Mayfield", "Burrow", "Fields", "Brady", "Cousins"]

def plot_hist(matrix, title, filename, out_folder):
    values = matrix.to_numpy(dtype=float)
    values = values[np.triu_indices_from(values, k=1)]
    values = values[~np.isnan(values)]
    if len(values) == 0:
        return
    plt.figure()
    plt.hist(values, bins=10)
    plt.title(title)
    plt.xlabel("Distance")
    plt.ylabel("Frequency")
    plt.savefig(os.path.join(out_folder, filename), dpi=300, bbox_inches='tight')
    plt.close()

for player_name in player_names:

    player_files = [
        f for f in os.listdir(folder)
        if player_name in f and f.endswith(".csv")
    ]

    if not player_files:
        continue

    #compute persistence diagrams
    player_results = {}
    for filename in player_files:
        filepath = os.path.join(folder, filename)
        year = filename.split("_")[-1].replace(".csv", "")
        try:
            player_results[year] = process_file(filepath)
        except Exception:
            pass

    if len(player_results) < 2:
        continue

    years_found = sorted(player_results.keys())

    bn_H0 = pd.DataFrame(index=years_found, columns=years_found, dtype=float)
    bn_H1 = pd.DataFrame(index=years_found, columns=years_found, dtype=float)
    ws_H0 = pd.DataFrame(index=years_found, columns=years_found, dtype=float)
    ws_H1 = pd.DataFrame(index=years_found, columns=years_found, dtype=float)

    #compute distances of player with themselves from year to year
    for i, y1 in enumerate(years_found):
        H0_1, H1_1 = player_results[y1]
        for j in range(i, len(years_found)):
            y2 = years_found[j]
            H0_2, H1_2 = player_results[y2]
            try:
                bn0 = bottleneck_distance(H0_1, H0_2)
                bn1 = bottleneck_distance(H1_1, H1_2)
                ws0 = wasserstein_distance(H0_1, H0_2, order=1., internal_p=2., keep_essential_parts=False)
                ws1 = wasserstein_distance(H1_1, H1_2, order=1., internal_p=2., keep_essential_parts=False)
                bn_H0.loc[y1, y2] = bn_H0.loc[y2, y1] = bn0
                bn_H1.loc[y1, y2] = bn_H1.loc[y2, y1] = bn1
                ws_H0.loc[y1, y2] = ws_H0.loc[y2, y1] = ws0
                ws_H1.loc[y1, y2] = ws_H1.loc[y2, y1] = ws1
            except Exception:
                pass

    #save results
    output_folder = f"{player_name}_career_analysis"
    os.makedirs(output_folder, exist_ok=True)
    bn_H0.to_csv(os.path.join(output_folder, f"{player_name}_H0_bottleneck.csv"))
    bn_H1.to_csv(os.path.join(output_folder, f"{player_name}_H1_bottleneck.csv"))
    ws_H0.to_csv(os.path.join(output_folder, f"{player_name}_H0_wasserstein.csv"))
    ws_H1.to_csv(os.path.join(output_folder, f"{player_name}_H1_wasserstein.csv"))

    #plot histograms
    plot_hist(bn_H0, f"{player_name} H0 Bottleneck Distances", f"{player_name}_H0_bn_hist.png", output_folder)
    plot_hist(bn_H1, f"{player_name} H1 Bottleneck Distances", f"{player_name}_H1_bn_hist.png", output_folder)
    plot_hist(ws_H0, f"{player_name} H0 Wasserstein Distances", f"{player_name}_H0_ws_hist.png", output_folder)
    plot_hist(ws_H1, f"{player_name} H1 Wasserstein Distances", f"{player_name}_H1_ws_hist.png", output_folder)


  Notable H0 Wasserstein Distances Across All Years 
2018: Brady vs Eli_Manning = 12.9789
2018: Brady vs Keenum = 13.5972
2018: Brady vs Rivers = 14.3667
2018: Brady vs Stafford = 8.8018
2018: Cousins vs Keenum = 14.1401
2018: Cousins vs Ryan = 10.8202
2018: Eli_Manning vs Rivers = 4.6811
2018: Eli_Manning vs Stafford = 6.4785
2018: Keenum vs Ryan = 14.2032
2018: Keenum vs Stafford = 12.9746
2018: Mahomes vs Watson = 13.5191
2018: Rivers vs Stafford = 8.8133
2018: Smith vs Wentz = 14.8363
2019: Brady vs Goff = 13.2133
2019: Brady vs Jackson = 213.3585
2019: Brady vs Mayfield = 13.8903
2019: Brees vs Flacco = 12.9461
2019: Brees vs Jackson = 232.0999
2019: Brissett vs Jackson = 193.3759
2019: Brissett vs Wentz = 14.4218
2019: Carr vs Garoppolo = 14.6487
2019: Carr vs Jackson = 232.5067
2019: Carr vs Rivers = 6.8273
2019: Cousins vs Jackson = 180.4898
2019: Cousins vs Minshew = 13.4432
2019: Dalton vs Jackson = 201.3214
2019: Darnold vs Jackson = 185.8383
2019: Fitzpatrick vs Minshew = 